# PyJST configurable CuPy GPU validation demo

Select a Colab GPU runtime. Adjust the next cell, then run the solve to export high-resolution pressure contours, convergence summary, and exact run metadata.

In [ ]:
# Install the current PyJST source and optional plotting/CUDA dependencies.
%pip install -q --upgrade --force-reinstall "git+https://github.com/VishalKandala/PyJST.git" "matplotlib>=3.7" "cupy-cuda12x>=13.0"


In [ ]:
from dataclasses import replace
from pathlib import Path
import importlib
import json
import shutil

import cupy as cp
from pyjst import CompressionCornerCase, JSTParameters, freestream_initial_state, solve
import pyjst.postprocess as postprocess
importlib.reload(postprocess)
from pyjst.postprocess import save_pressure_contour_figure, save_solution_figure

device = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)["name"].decode()
print(f"CuPy {cp.__version__} on {device}")
output_dir = Path("pyjst-validation-results")
output_dir.mkdir(exist_ok=True)


## Case and convergence controls

In [ ]:
# Configure this validation demonstrator before running the solve cell.
nx, ny = 160, 80
mach = 2.0
deflection_degrees = 10.0
residual_tolerance = 1.0e-6
max_iterations = 10_000
cfl = 0.4
cfl_initial = 0.05
cfl_ramp_iterations = 100
contour_levels = 32


## Run and export contour validation artifacts

In [ ]:
definition = CompressionCornerCase(
    nx=nx, ny=ny, mach=mach, deflection_degrees=deflection_degrees,
)
case = replace(
    definition.solver_case(),
    numerics=JSTParameters(
        cfl=cfl, cfl_initial=cfl_initial, cfl_ramp_iterations=cfl_ramp_iterations,
        residual_tolerance=residual_tolerance, max_iterations=max_iterations,
    ),
)
grid = definition.grid()
result = solve(freestream_initial_state(grid, case), grid, case, backend="cupy")

status = "converged" if result.converged else "iteration cap reached without convergence"
print(f"GPU validation run: {status}")
print(f"iterations: {result.iterations:,} / {max_iterations:,}")
print(f"normalized residual: {result.residual_history[-1]:.6e}")

contour_png = output_dir / "compression-corner-pressure-contours.png"
summary_png = output_dir / "compression-corner-summary.png"
save_pressure_contour_figure(
    result, grid, case, contour_png, levels=contour_levels,
    title=f"CuPy GPU pressure contours: {result.iterations:,} iterations, residual {result.residual_history[-1]:.2e}", dpi=300,
)
save_solution_figure(
    result, grid, case, summary_png,
    title=f"CuPy GPU convergence summary: {result.iterations:,} iterations", dpi=300,
)
(output_dir / "validation-summary.json").write_text(json.dumps({
    "gpu": device, "cupy": cp.__version__, "grid": [nx, ny],
    "mach": mach, "deflection_degrees": deflection_degrees,
    "iteration_cap": max_iterations, "iterations": result.iterations,
    "residual_tolerance": residual_tolerance,
    "normalized_residual": float(result.residual_history[-1]),
    "converged": result.converged,
}, indent=2) + "\n")

from IPython.display import Image, display
display(Image(filename=str(contour_png)))
archive = shutil.make_archive("pyjst-validation-results", "zip", root_dir=output_dir)
from google.colab import files
files.download(archive)
